In [ ]:
!pip install deepface mtcnn kaggle tensorflow opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'sreelakshmivit05'
os.environ['KAGGLE_KEY'] = 'KGAT_3000b193ac75966a273c219b222f89cf'
print("Kaggle ready!")

Kaggle ready!


In [ ]:
import os

# LFW Dataset
!kaggle datasets download -d atulanandjha/lfwpeople -p /content/lfw --unzip
print("LFW downloaded!")

# Yawn Eye Dataset
!kaggle datasets download -d serenaraju/yawn-eye-dataset-new -p /content/yawn --unzip
print("Yawn Eye Dataset downloaded!")

Dataset URL: https://www.kaggle.com/datasets/atulanandjha/lfwpeople
License(s): GNU Lesser General Public License 3.0
100% 232M/232M [00:02<00:00, 107MB/s]

LFW downloaded!
Dataset URL: https://www.kaggle.com/datasets/serenaraju/yawn-eye-dataset-new
License(s): unknown
100% 161M/161M [00:01<00:00, 115MB/s]

Yawn Eye Dataset downloaded!


In [ ]:
import zipfile, os

# Fix LFW
for f in os.listdir('/content/lfw'):
    if f.endswith('.zip'):
        print(f"Extracting {f}...")
        with zipfile.ZipFile(f'/content/lfw/{f}', 'r') as z:
            z.extractall('/content/lfw')
        print("LFW extracted!")

# Fix Yawn
for f in os.listdir('/content/yawn'):
    if f.endswith('.zip'):
        print(f"Extracting {f}...")
        with zipfile.ZipFile(f'/content/yawn/{f}', 'r') as z:
            z.extractall('/content/yawn')
        print("Yawn extracted!")

# Check result
for root, dirs, files in os.walk('/content/lfw'):
    level = root.replace('/content/lfw', '').count(os.sep)
    if level < 3:
        print(f"{'  '*level}{root} ({len(files)} files)")

/content/lfw (4 files)


In [ ]:
import os
for f in os.listdir('/content/lfw'):
    print(f)

lfw-funneled.tgz
pairsDevTrain.txt
pairs.txt
pairsDevTest.txt


In [ ]:
import tarfile

print("Extracting...")
with tarfile.open('/content/lfw/lfw-funneled.tgz', 'r:gz') as tar:
    tar.extractall('/content/lfw')
print("Done!")

# Check
import os
folders = os.listdir('/content/lfw')
print(folders)

Extracting...


/tmp/ipykernel_4794/1100820195.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/lfw')


Done!
['lfw-funneled.tgz', 'lfw_funneled', 'pairsDevTrain.txt', 'pairs.txt', 'pairsDevTest.txt']


In [ ]:
import os, shutil

CUSTOM_PATH = '/content/drive/MyDrive/FaceAttendance/custom_dataset'
LFW_PATH    = '/content/lfw/lfw_funneled'

for person in os.listdir(CUSTOM_PATH):
    src  = os.path.join(CUSTOM_PATH, person)
    if not os.path.isdir(src):
        continue
    dest = os.path.join(LFW_PATH, person)
    os.makedirs(dest, exist_ok=True)
    count = 0
    for img in os.listdir(src):
        shutil.copy(os.path.join(src, img), os.path.join(dest, img))
        count += 1
    print(f"{person}: {count} photos added to LFW")

print("\nMerge complete!")

sreelakshmi: 114 photos added to LFW
anushka: 40 photos added to LFW
pournami: 30 photos added to LFW
jianna: 90 photos added to LFW
gowri: 30 photos added to LFW

Merge complete!


In [ ]:
import cv2
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os

# OpenCV face detector — much faster than MTCNN
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

LFW_PATH = '/content/lfw/lfw_funneled'
faces, labels = [], []
MIN_IMAGES = 20

people = [p for p in os.listdir(LFW_PATH)
          if os.path.isdir(os.path.join(LFW_PATH, p)) and
          len(os.listdir(os.path.join(LFW_PATH, p))) >= MIN_IMAGES]

print(f"People with 20+ images: {len(people)}")

for person in people:
    folder = os.path.join(LFW_PATH, person)
    for img_file in os.listdir(folder):
        img_path = os.path.join(folder, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        detected = face_cascade.detectMultiScale(gray, 1.1, 4)
        if len(detected) > 0:
            x, y, w, h = detected[0]
            face = cv2.cvtColor(img[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
            face = cv2.resize(face, (224, 224)) / 255.0
            faces.append(face)
            labels.append(person)

faces  = np.array(faces)
labels = np.array(labels)

le          = LabelEncoder()
labels_enc  = le.fit_transform(labels)

X_train, X_test, y_train, y_test = train_test_split(
    faces, labels_enc, test_size=0.2, random_state=42)

print(f"Face Recognition Data Ready!")
print(f"   Train: {X_train.shape} | Test: {X_test.shape}")
print(f"   Classes: {len(le.classes_)}")

People with 20+ images: 67
Face Recognition Data Ready!
   Train: (2648, 224, 224, 3) | Test: (663, 224, 224, 3)
   Classes: 67


In [ ]:
import os
for root, dirs, files in os.walk('/content/yawn'):
    level = root.replace('/content/yawn', '').count(os.sep)
    if level < 4:
        print(f"{'  '*level}{root} ({len(files)} files)")

/content/yawn (0 files)
  /content/yawn/dataset_new (0 files)
    /content/yawn/dataset_new/train (0 files)
      /content/yawn/dataset_new/train/Closed (617 files)
      /content/yawn/dataset_new/train/Open (617 files)
      /content/yawn/dataset_new/train/yawn (617 files)
      /content/yawn/dataset_new/train/no_yawn (616 files)
    /content/yawn/dataset_new/test (0 files)
      /content/yawn/dataset_new/test/Closed (109 files)
      /content/yawn/dataset_new/test/Open (109 files)
      /content/yawn/dataset_new/test/yawn (106 files)
      /content/yawn/dataset_new/test/no_yawn (109 files)


In [ ]:
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import os

YAWN_PATH = '/content/yawn/dataset_new'

label_map = {
    'Closed': 0,   # distracted
    'Open': 1,     # attentive
    'yawn': 0,     # distracted
    'no_yawn': 1   # attentive
}

eye_imgs, eye_labels = [], []

for split in ['train', 'test']:
    for folder_name, label in label_map.items():
        folder_path = os.path.join(YAWN_PATH, split, folder_name)
        if not os.path.exists(folder_path):
            continue
        for img_file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_file)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))
            img = img / 255.0
            eye_imgs.append(img)
            eye_labels.append(label)

eye_imgs = np.array(eye_imgs)
eye_labels = np.array(eye_labels)

X_attn_train, X_attn_test, y_attn_train, y_attn_test = train_test_split(
    eye_imgs, eye_labels, test_size=0.2, random_state=42)

print(f"Attention Data Ready!")
print(f"   Train: {X_attn_train.shape} | Test: {X_attn_test.shape}")
print(f"   Attentive: {sum(eye_labels==1)} | Distracted: {sum(eye_labels==0)}")

Attention Data Ready!
   Train: (2320, 64, 64, 3) | Test: (580, 64, 64, 3)
   Attentive: 1451 | Distracted: 1449


In [ ]:
import pickle, os
import numpy as np

SAVE_PATH = '/content/drive/MyDrive/FaceAttendance/data'
os.makedirs(SAVE_PATH, exist_ok=True)

# Save face recognition data
np.save(f'{SAVE_PATH}/X_face_train.npy', X_train)
np.save(f'{SAVE_PATH}/X_face_test.npy',  X_test)
np.save(f'{SAVE_PATH}/y_face_train.npy', y_train)
np.save(f'{SAVE_PATH}/y_face_test.npy',  y_test)

# Save attention data
np.save(f'{SAVE_PATH}/X_attn_train.npy', X_attn_train)
np.save(f'{SAVE_PATH}/X_attn_test.npy',  X_attn_test)
np.save(f'{SAVE_PATH}/y_attn_train.npy', y_attn_train)
np.save(f'{SAVE_PATH}/y_attn_test.npy',  y_attn_test)

# Save label encoder
with open(f'{SAVE_PATH}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("All data saved to Google Drive!")
print(f"   Location: {SAVE_PATH}")

All data saved to Google Drive!
   Location: /content/drive/MyDrive/FaceAttendance/data


In [ ]:
for person in ['anushka', 'sreelakshmi', 'pournami', 'jianna', 'gowri']:
    path = os.path.join('/content/lfw/lfw_funneled', person)
    if os.path.exists(path):
        print(f"{person}: {len(os.listdir(path))} photos")
    else:
        print(f"{person}: NOT FOUND")

anushka: 40 photos
sreelakshmi: 114 photos
pournami: 30 photos
jianna: 90 photos
gowri: 30 photos
